In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:

# i will import tourch because its not imported above
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

# 1. Convert Numpy arrays to PyTorch Tensors
# Note: i will use float
X_train_tensor = torch.tensor(X_train).float()
y_train_tensor = torch.tensor(y_train).float().view(-1, 1) # Reshape to (N, 1)

X_test_tensor = torch.tensor(X_test).float()
y_test_tensor = torch.tensor(y_test).float().view(-1, 1)

print("Tensors created.")

In [ ]:
 #Create Datasets:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print("Datasets created.")

In [ ]:
# 3. Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print("DataLoaders created.")


In [ ]:
# 4. Print shape of one batch
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Batch Image Shape: {images.shape}") # Should be [32, 3, H, W]
print(f"Batch Label Shape: {labels.shape}") # Should be [32, 1]


In [ ]:
# Get one batch of images and labels
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 5. Display sample images
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    # Get image and transpose: (Channels, Height, Width) -> (Height, Width, Channels)
    img_to_show = images[i].permute(1, 2, 0).numpy()

    plt.imshow(img_to_show)
    plt.title(f"Age: {int(labels[i].item())}")
    plt.axis('off')
plt.show()

In [ ]:
# Task 1: Write your model class here:
class AgePredictor(nn.Module):
    def __init__(self, input_dim):
        super(AgePredictor, self).__init__()


        # We reduce the size gradually
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Flatten the image tensor
        x = x.view(x.size(0), -1)

        # Pass through layers
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))

        # Output layer
        x = self.fc4(x)
        return x

In [ ]:
# Task 2: Write your training loop here:
# Task 2: Write your training loop here:
def train_model(model, loader, criterion, optimizer, device):
    model.train() # Set to train mode
    running_loss = 0.0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # 1. Zero gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # 3. Backward pass
        loss.backward()

        # 4. Optimizer step
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
# Task 3: Write your validation loop here:
def validate_model(model, loader, criterion, device):
    model.eval() # Set to evaluation mode
    running_loss = 0.0

    with torch.no_grad(): # Disable gradient calculation
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Calculate input dimension (C * H * W)
sample_batch, _ = next(iter(train_loader))
input_dim = sample_batch.shape[1] * sample_batch.shape[2] * sample_batch.shape[3]

model = AgePredictor(input_dim).to(device)

# Loss Function: MSE for Regression
criterion = nn.MSELoss()

# Optimizer: Adam
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
train_losses = []
val_losses = []

print("Starting training...")

for epoch in range(num_epochs):
    train_loss = train_model(model, train_loader, criterion, optimizer, device)
    val_loss = validate_model(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:
# Task 1: Plot loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
model.eval()
inputs, targets = next(iter(test_loader))
inputs, targets = inputs.to(device), targets.to(device)

# predictions
with torch.no_grad():
    predictions = model(inputs)

# we study that must have data in cpu if u want to plot them)
inputs = inputs.cpu()
targets = targets.cpu()
predictions = predictions.cpu()

# i will Plot 5 images here
plt.figure(figsize=(15, 5))
for i in range(5):
    plt.subplot(1, 5, i+1)
    img_to_show = inputs[i].permute(1, 2, 0).numpy()

    plt.imshow(img_to_show)
    plt.title(f"True: {int(targets[i].item())}\nPred: {predictions[i].item():.1f}")
    plt.axis('off')

plt.suptitle("Model Predictions vs Actual Age")
plt.tight_layout()
plt.show()